# ParamISP demo - Google Colab

Demo for **ParamISP: Learned Forward and Inverse ISPs using Camera Parameters** (CVPR 2024).

Two directions: **sRGB -> RAW** (inverse ISP) and **RAW -> sRGB** (forward ISP), controlled by camera parameters (ISO, exposure, focal length, f-number, white balance, colour matrix).

**No GPU is required** - the model is only ~0.7M parameters and runs on CPU. To use a free GPU anyway: **Runtime > Change runtime type > T4 GPU**.

This notebook runs the model and shows the results inline. For a hosted web UI, run the Streamlit app in `streamlit_app/` (see the README).

In [ ]:
!git clone https://github.com/gliderest/paramISP.git
%cd paramISP

In [ ]:
# Runtime dependencies (torch is already provided by Colab).
!pip install -q scipy pillow tifffile exifread matplotlib toolz pyyaml \
  python-dotenv rawpy gdown "kornia>=0.7,<0.9"

In [ ]:
import os
os.environ["PYTHONPATH"] = "."

import numpy as np
from paramisp_demo import dataio, inference, weights

device = inference.resolve_device()
print("device:", device)

WB = (2.0, 1.0, 1.5)
srgb = dataio.load_srgb("assets/example_srgb.jpg")

# sRGB -> RAW (inverse ISP). Checkpoints download on first use.
inv = inference.load_model(weights.ensure_weights("inverse", "D7000"), True, device)[0]
raw = inference.predict_srgb_to_raw(
    inv, srgb, device, camera_model="D7000",
    bayer_pattern="RGGB", white_balance=WB,
)
print("predicted RAW:", raw.shape, float(raw.min()), float(raw.max()))

In [ ]:
# RAW -> sRGB (forward ISP), then show input | predicted RAW | reconstruction.
from PIL import Image

preview = dataio.visualize_raw(raw, "RGGB", WB, camera_model="D7000")

fwd = inference.load_model(weights.ensure_weights("forward", "D7000"), False, device)[0]
rec = inference.predict_raw_to_srgb(
    fwd, raw, device, camera_model="D7000",
    bayer_pattern="RGGB", white_balance=WB,
)

strip = np.concatenate([srgb, preview, np.clip(rec, 0.0, 1.0)], axis=1)
Image.fromarray((np.clip(strip, 0.0, 1.0) * 255).astype(np.uint8)).save("paramisp_result.png")
display(Image.open("paramisp_result.png"))